# Portfolio Optimization with PCE (Pauli Correlation Encoding)

> 🚀 **Don't choke your local machine.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why PCE?

PCE uses **polynomial encoding** to compress QUBO variables logarithmically. Where standard QAOA needs n qubits for n binary variables, PCE achieves the same with **far fewer qubits** — making it ideal for NISQ hardware.

Combined with QoroService, PCE portfolios can scale to hundreds of assets while keeping qubit counts manageable.

## Step 0 — Install & Authenticate

```bash
pip install qoro-divi numpy pandas
```

In [ ]:
import os
import time

import numpy as np
import dimod
import pennylane as qml

# Set your API key (get one free at https://dash.qoroquantum.net)
# Create a .env file at the root and set the QORO_API_TOKEN

from divi.backends import QoroService, ParallelSimulator, JobConfig
from divi.qprog import PCE, GenericLayerAnsatz
from divi.qprog.optimizers import PymooOptimizer, PymooMethod

from utils import build_qubo_matrices_for_all_partitions, evaluate_solution

---

## Phase 1 — Small Portfolio with PCE (Local)

8 binary asset variables compressed into fewer qubits via polynomial encoding.

In [ ]:
# Generate synthetic financial data
np.random.seed(42)
n_assets = 8

returns = np.random.uniform(0.01, 0.1, n_assets)
A = np.random.randn(n_assets, n_assets) * 0.1
covariance = A @ A.T + np.eye(n_assets) * 0.01

print(f"📊 Portfolio: {n_assets} assets")
print(f"   Returns range: [{returns.min():.4f}, {returns.max():.4f}]")

In [ ]:
LAMBDA_PARAM = 0.75

qubo_matrices = build_qubo_matrices_for_all_partitions(
    {0: returns}, {0: covariance}, lambda_param=LAMBDA_PARAM
)
qubo_matrix = qubo_matrices[0]
bqm = dimod.BinaryQuadraticModel(qubo_matrix, "BINARY")
print(f"QUBO: {len(bqm.variables)} binary variables")

In [ ]:
local_backend = ParallelSimulator(shots=10_000)
optim = PymooOptimizer(PymooMethod.DE, population_size=30)

print("🚀 Running PCE-VQE locally...")
t0 = time.time()

ansatz = GenericLayerAnsatz(
    gate_sequence=[qml.RY, qml.RZ],
    entangler=qml.CNOT,
    entangling_layout="all-to-all",
)

pce = PCE(
    bqm,
    ansatz=ansatz,
    n_layers=3,
    optimizer=optim,
    max_iterations=15,
    backend=local_backend,
    encoding_type="poly",
)
pce.run()

local_time = time.time() - t0
print(f"✅ PCE complete in {local_time:.1f}s")
print(f"   Qubits used: {pce.n_qubits} (compressed from {n_assets} binary variables)")

In [ ]:
pce_solution = np.array(pce.solution, dtype=int)
print(f"PCE solution: {pce_solution}")
print(f"Assets selected: {np.where(pce_solution == 1)[0]}")

evaluate_solution(qubo_matrix, pce_solution, returns, covariance)

---

## Phase 2 — Scale Up with QoroService

For larger portfolios, swap one line:

```python
cloud_backend = QoroService(job_config=JobConfig(shots=10_000))
```

PCE's qubit compression + QoroService's parallel execution = scalable portfolio optimization.

👉 **[Get your free API key →](https://dash.qoroquantum.net)**

---

## Ready for Real-World Portfolios?

PCE compresses qubits logarithmically. QoroService parallelizes execution. Together, they scale to hundreds of assets.

👉 **[Get your free API key at dash.qoroquantum.net](https://dash.qoroquantum.net)** — $100 in credits, no credit card required.